# TP — Détection de beaconing C2

## Objectif

On dispose d'un dataframe contenant les logs de connexions sortantes sur un réseau d'entreprise.

Certains hôtes sont compromis et contactent régulièrement un serveur « Commande et contrôle » (C2). Le but du TD est de :
- identifier les hôtes compromis ;
- extraire les informations de compromission (IOC).

**Consignes :** pour chaque encadré « ✏️ Exercice », complétez le code
demandé puis exécutez la cellule de vérification qui la suit, ou rédigez la
réponse à la question posée (exercices sans code).

In [ ]:
import hashlib
import random
import statistics
from datetime import datetime, timedelta

import pandas as pd

pd.set_option("display.max_rows", 100)  # afficher les tableaux en entier


def empreinte(valeur) -> str:
    """Empreinte courte d'une valeur : vérifie un résultat sans le dévoiler."""
    return hashlib.sha256(repr(valeur).encode()).hexdigest()[:12]

## Préambule : quelques outils pandas

Le TD utilise les méthodes suivantes. On les illustre ici sur un petit dataframe `mini`.

| méthode | rôle |
| --- | --- |
| `df.sort_values("col")` | trier les lignes selon une colonne (`ascending=False` : décroissant) |
| `df.iloc[0]` | première ligne, **par position** (sur une série : première valeur) |
| `serie.index[0]` | **étiquette d'index** de la première ligne |
| `serie.isin(ensemble)` | booléen par ligne : la valeur est-elle dans l'ensemble ? |
| `~masque` | négation d'un masque booléen (« **pas** dans l'ensemble ») |
| `df[(cond1) & (cond2)]` | lignes vérifiant **les deux** conditions (parenthèses obligatoires, pas de `and`) |
| `df.drop_duplicates("col")` | garder une seule ligne par valeur de `col` |
| `df.groupby("col")["col2"].count()` | série indexée par `col` : nombre de valeurs de `col2` par groupe (de même : `.sum()`) |
| `df.groupby("col")["col2"].nunique()` | idem, nombre de valeurs **distinctes** |
| `df.groupby(["c1", "c2"])["col2"].count()` | idem, groupes = couples `(c1, c2)` (index à deux niveaux) |
| `df.groupby("col").agg(nom=("col2", f))` | dataframe : une ligne par groupe, une colonne `nom` = `f` appliquée à `col2` (`f` peut être `"count"`, `"nunique"`, `list`…) |

In [ ]:
mini = pd.DataFrame({
    "src_ip": ["10.0.0.1", "10.0.0.1", "10.0.0.2", "10.0.0.2", "10.0.0.2", "10.0.0.3"],
    "domain": ["google.com", "ntp.org", "google.com", "google.com", "evil.xyz", "google.com"],
    "bytes_out": [1200, 48, 800, 3000, 512, 450],
})
mini

### Trier, filtrer, dédoublonner

Trier par taille envoyée, de la plus grande à la plus petite :

In [ ]:
mini.sort_values("bytes_out", ascending=False)

La première ligne de ce tri (la plus grosse connexion) : `.iloc[0]` la donne
**par position** (sur un dataframe : toute la ligne ; sur une série : la valeur),
`.index[0]` donne son **label d'index**.

In [ ]:
tri = mini.sort_values("bytes_out", ascending=False)
print("label d'index :", tri.index[0])
tri.iloc[0]

Filtrer par appartenance : `isin` garde les lignes dont le domaine est dans
l'ensemble, `~` inverse le masque (on **retire** ces domaines).

In [ ]:
connus = {"google.com", "ntp.org"}
print(mini["domain"].isin(connus).tolist())
mini[~mini["domain"].isin(connus)]

Combiner deux conditions : `&` (« et »), `|` (« ou »). Chaque condition doit
être entre **parenthèses**, et `and` / `or` ne fonctionnent pas sur des colonnes
(erreur « *truth value of a Series is ambiguous* »).

In [ ]:
mini[(mini["src_ip"] == "10.0.0.2") & (mini["bytes_out"] > 600)]

Garder une seule ligne par domaine (la première rencontrée) :

In [ ]:
mini.drop_duplicates("domain")

### `groupby` : résumer par groupe

TODO: vérifier si on peut simplifier cette partie: est ce que `df.groupby("col")["col2"].f()` est utilisé dans le td ?

`df.groupby("col")["col2"].f()` applique `f` aux valeurs de `col2` de chaque
groupe : le résultat est une **série**, indexée par la clé de groupe.

Par domaine : nombre de connexions (`count`) et nombre d'hôtes **distincts**
(`nunique`). `google.com` reçoit 4 connexions, mais de 3 hôtes seulement.

In [ ]:
print(mini.groupby("domain")["src_ip"].count())
print()
print(mini.groupby("domain")["src_ip"].nunique())

Grouper sur **deux** colonnes : on passe une **liste** de colonnes. Chaque groupe
est un couple `(src_ip, domain)`, et l'index du résultat a deux niveaux.

In [ ]:
mini.groupby(["src_ip", "domain"])["bytes_out"].count()

Total d'octets envoyés par hôte, trié : sur le résultat d'un `groupby`, l'index est la
clé de groupe, donc `.index[0]` répond à « quel hôte ? » et `.iloc[0]` à
« combien ? ».

In [ ]:
total_octets = mini.groupby("src_ip")["bytes_out"].sum().sort_values(ascending=False)
print(total_octets)
print()
print("hôte :", total_octets.index[0], "| total :", total_octets.iloc[0])

`agg` avec des sorties nommées renvoie un **dataframe** : ici, par hôte, la liste
des domaines contactés et le nombre de connexions.

In [ ]:
mini.groupby("src_ip").agg(domaines=("domain", list), n=("domain", "count"))

## Préambule : chargement des données

Le fichier `logs_reseau.csv` contient les connexions sortantes observées pendant
12 h (08:00 → 20:00) sur un réseau d'entreprise.

| colonne | description |
| --- | --- |
| `timestamp` | instant de la connexion |
| `src_ip` | hôte source (interne) |
| `dst_ip` | IP de destination |
| `domain` | domaine contacté (nom DNS) |
| `dst_port` | port de destination |
| `bytes_out` | octets envoyés |

In [ ]:
df_logs_reseau = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])
print(df_logs_reseau.shape)
df_logs_reseau.head()

In [ ]:
df_logs_reseau.info()

## 1. Beaconing « classique »

Un implant « beacon » rappelle son serveur C2 à intervalle **régulier**. Objectif :
repérer le couple (hôte, domaine) au rythme le plus régulier.

Pour cela, on calcule le « coefficient de variation » (CV) des intervalles de temps entre requêtes successives :

   - `CV ≈ 0` → intervalle quasi constant (beacon *ou* tâche planifiée) ;
   - `CV` élevé → navigation humaine, irrégulière.

Étant donnée une liste **triée** de timestamps de requêtes, le coefficient de variation se calcule ainsi :
- on construit la liste des intervalles de temps, **en secondes**, entre chaque couple de requêtes successives ;
- sur cette liste, `CV = écart-type / moyenne`.

**Méthode, pas à pas :**

1. **Regrouper en listes** : fabriquer un DataFrame où chaque ligne est un couple
   `(src_ip, domain)` et où l'on range ses timestamps dans une **liste** (triée).
2. **Transformer chaque liste en écarts** : une petite fonction Python
   (`get_diffs`) qui renvoie les temps inter-arrivées, en secondes.
3. **Résumer en CV** : calculer le coefficient de variation pour chacune de ces listes.

### 1.1 Calculer un CV sur un exemple

Pour une liste d'écarts $x_1, \dots, x_n$ :

$$CV = \frac{s}{m}$$

où :

- $m$ est la **moyenne** des écarts (`statistics.mean`) ;
- $s$ est leur **écart-type** (`statistics.stdev`), qui mesure leur dispersion
  autour de $m$ : il est nul si tous les écarts sont égaux, et d'autant plus
  grand qu'ils s'éloignent de la moyenne.

Diviser par $m$ rend le CV sans unité : on peut comparer des rythmes rapides et lents.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1a</legend>

Complétez `get_diffs` : elle renvoie, dans l'ordre chronologique, les écarts en
secondes entre instants successifs d'une liste de timestamps (liste vide s'il y a
moins de deux instants). Elle sert dans toute la partie 1.

</fieldset>

In [ ]:
def get_diffs(timestamps: list[datetime]) -> list[float]:
    """
    Prend en entrée une liste de timestamps et renvoie la liste des écarts entre timestamps successifs.

    Notes:
    - La liste `timestamps` n'est pas forcément donnée triée en entrée.
    - Renvoie les écarts en secondes (cf. méthode `.total_seconds()` de `timedelta`).
    """
    # BEGIN
    # raise NotImplementedError
    tri = sorted(timestamps)
    res = []
    for i in range(len(tri) - 1):
        res.append((tri[i + 1] - tri[i]).total_seconds())
    return res
    # END

In [ ]:
t0 = datetime(2026, 10, 14, 8, 0, 0)
assert get_diffs([t0, t0 + timedelta(seconds=60), t0 + timedelta(seconds=120)]) == [60.0, 60.0]
assert get_diffs([t0 + timedelta(seconds=120), t0, t0 + timedelta(seconds=60)]) == [60.0, 60.0]
assert get_diffs([t0]) == []
assert get_diffs([]) == []
print("ex 1a OK")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1b</legend>

Trois couples (hôte, domaine) ont été observés sur une courte fenêtre :

| couple | instants des connexions |
| --- | --- |
| A : `10.0.0.21 → cdn-sync.example.net` | 08:00:00, 08:01:00, 08:02:00, 08:03:00 |
| B : `10.0.0.22 → telemetry.example.net` | 08:00:00, 08:00:55, 08:01:55, 08:03:00 |
| C : `10.0.0.23 → news.example.org` | 08:00:00, 08:01:40, 08:11:40, 08:30:00 |

Ces instants sont fournis dans le dictionnaire `connexions` (clés `"A"`, `"B"`,
`"C"`). En utilisant `get_diffs`, construisez deux dictionnaires de mêmes clés :

- `ecarts` : la liste des écarts, **en secondes**, entre connexions successives ;
- `cv` : le coefficient de variation de cette liste (valeur exacte, sans arrondi).

</fieldset>

In [ ]:
JOUR = "2026-10-14 "
connexions = {
    "A": [datetime.fromisoformat(JOUR + h) for h in ["08:00:00", "08:01:00", "08:02:00", "08:03:00"]],
    "B": [datetime.fromisoformat(JOUR + h) for h in ["08:00:00", "08:00:55", "08:01:55", "08:03:00"]],
    "C": [datetime.fromisoformat(JOUR + h) for h in ["08:00:00", "08:01:40", "08:11:40", "08:30:00"]],
}

# BEGIN
## Rappels :
## - `get_diffs(instants)` donne la liste des écarts en secondes ;
## - `statistics.mean(liste)` / `statistics.stdev(liste)` : moyenne et écart-type.
## Une compréhension de dictionnaire suffit pour chacun :
## `{cle: f(valeur) for cle, valeur in d.items()}`.
# ecarts = ...
# cv = ...
ecarts = {nom: get_diffs(instants) for nom, instants in connexions.items()}
cv = {nom: statistics.stdev(liste) / statistics.mean(liste) for nom, liste in ecarts.items()}
# END

In [ ]:
assert set(ecarts) == set(cv) == set(connexions), "clés attendues : A, B, C"
somme_carres = float(sum(x**2 for liste in ecarts.values() for x in liste))
assert empreinte(somme_carres) == "9eae21eb8979", "écarts incorrects"
assert empreinte(round(sum(cv.values()), 4)) == "0abd61293f58", "CV incorrects"
print("ex 1b OK")

Après la vérification, la cellule suivante affiche les écarts et les CV obtenus.

In [ ]:
for nom in connexions:
    print(f"{nom} : écarts = {ecarts[nom]}  CV = {cv[nom]:.2f}")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1c</legend>

On lit les résultats ci-dessus.

1. Classez les trois couples du plus régulier au moins régulier. Lequel ressemble à de la navigation humaine ?
2. Peut-on conclure que A est un beacon malveillant ? Que dire de B ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

Écarts — A : `60, 60, 60` ; B : `55, 60, 65` ; C : `100, 600, 1100`.

- A : tous les écarts sont égaux → $s = 0$, **CV = 0**.
- B : $m = 60$, $s = 5$ → **CV = 5/60 ≈ 0,08**.
- C : $m = 600$, $s = 500$ → **CV = 500/600 ≈ 0,83**.

1. A (0) < B (0,08) < C (0,83). C, irrégulier, ressemble à de la navigation humaine.
2. Non : un CV nul dit seulement que le trafic est **périodique**. Une mise à
   jour, une synchronisation ou NTP sont aussi périodiques ; il faut croiser
   avec d'autres indices (domaine connu ou non, taille des messages…). B est
   presque aussi régulier que A : un léger aléa (*jitter*) de quelques secondes
   autour de 60 s ne suffit pas à masquer le rythme.

</fieldset>

### 1.2 Regrouper les timestamps par couple (src_ip, domain)

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Créez le dataframe `par_couple`, indexé par `(src_ip, domain)`, avec une colonne `timestamps` contenant la liste des timestamps de connexion de chaque couple (à partir de `df_logs_reseau`).

*Remarque :* `logs_reseau.csv` est déjà trié par `timestamp` : ces listes sont donc
automatiquement triées.

</fieldset>

In [ ]:
# BEGIN
## Suggestion :  afficher `df_logs_reseau` pour voir à quoi il ressemble;
## Utiliser `groupby` puis `agg`
# par_couple = ...
par_couple = (
    df_logs_reseau.groupby(["src_ip", "domain"])
    .agg(timestamps=("timestamp", list))
)
# END

In [ ]:
assert isinstance(par_couple, pd.DataFrame), "par_couple doit être un DataFrame (pas une Series)"
nb_couples = df_logs_reseau[["src_ip", "domain"]].drop_duplicates().shape[0]
assert par_couple.shape[0] == nb_couples
assert par_couple.index.names == ["src_ip", "domain"], "index attendu : (src_ip, domain)"
assert "timestamps" in par_couple.columns
print("ex 2 OK :", nb_couples, "couples (src_ip, domain) distincts")

### 1.3 Résumer en coefficient de variation

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3a</legend>

Ajoutez à `par_couple` deux colonnes :

- `diffs` : la liste des écarts de chaque couple (calculée avec `get_diffs`) ;
- `n` : le nombre d'écarts.

</fieldset>

In [ ]:
# BEGIN
## Indice : `df[c].map(f)` applique `f` à chaque élément de la colonne c
# par_couple["diffs"] = ...
# par_couple["n"] = ...
par_couple["diffs"] = par_couple["timestamps"].map(get_diffs)
par_couple["n"] = par_couple["diffs"].map(len)
# END

In [ ]:
assert {"timestamps", "diffs", "n"} <= set(par_couple.columns), f"colonnes attendues : timestamps, diffs, n ; obtenues : {list(par_couple.columns)}"
assert par_couple.shape[0] == nb_couples, "aucun couple ne doit être retiré"
assert (par_couple["diffs"].map(len) == par_couple["n"]).all(), "n doit être la longueur de diffs"
assert empreinte(int(par_couple["n"].sum())) == "511425f1125b", "écarts incorrects"
print("ex 3a OK")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3b</legend>

Créez le dataframe `cv_par_couple`, de colonnes `src_ip`, `domain`, `n`, `cv` (dans
cet ordre) : pour chaque couple de `par_couple` ayant au moins `N_MIN` écarts (un CV
sur 2 points ne veut rien dire), le coefficient de variation `cv` de sa liste d'écarts.

</fieldset>

In [ ]:
N_MIN = 20  # nombre minimal d'écarts pour qu'un CV ait un sens

# BEGIN
## Indice : `df[df["n"] >= N_MIN]` garde les lignes vérifiant la condition ;
## `statistics.mean(liste)` / `statistics.stdev(liste)` : moyenne et écart-type ;
## `df.reset_index()` transforme l'index (src_ip, domain) en colonnes.
# cv_par_couple = ...
cv_par_couple = par_couple[par_couple["n"] >= N_MIN].copy()
cv_par_couple["moyenne"] = cv_par_couple["diffs"].map(statistics.mean)
cv_par_couple["ecart_type"] = cv_par_couple["diffs"].map(statistics.stdev)
cv_par_couple["cv"] = cv_par_couple["ecart_type"] / cv_par_couple["moyenne"]
cv_par_couple = cv_par_couple.reset_index()[["src_ip", "domain", "n", "cv"]]
# END

In [ ]:
assert list(cv_par_couple.columns) == ["src_ip", "domain", "n", "cv"], f"colonnes attendues : ['src_ip', 'domain', 'n', 'cv'] ; obtenues : {list(cv_par_couple.columns)}"
assert (cv_par_couple["n"] >= N_MIN).all(), "les couples trop rares doivent être écartés"
assert len(cv_par_couple) == 63, "nombre de couples incorrect"
assert empreinte(round(float(cv_par_couple["cv"].sum()), 4)) == "666d90f3b1e5", "CV incorrects"
print("ex 3b OK")

On affiche ici tous les couples `(src_ip, domain)` retenus, triés par CV croissant.

In [ ]:
cv_par_couple.sort_values("cv")

### 1.4 Liste blanche : les faux positifs périodiques

Le tri par CV ci-dessus montre que les couples les plus réguliers ne sont pas
forcément malveillants : les mises à jour (Windows, antivirus, NTP…) contactent
elles aussi leur serveur à intervalle quasi constant — avec un petit jitter réseau
du même ordre que celui du beacon. Ce sont des **faux positifs**.

L'équipe SOC fournit la liste des domaines légitimes connus (liste blanche) :
on les écarte avant de chercher le beacon. Cette liste est **réutilisée dans la
suite du TD** chaque fois qu'il faut écarter le trafic légitime.

Domaines whitelistés : `windowsupdate.com`, `ntp.org`, `av-updates.example.net`,
`sync-backup.example.net`, `metrics-updates.example.net`.

In [ ]:
# Domaines légitimes périodiques (liste blanche fournie par le SOC).
# À réutiliser chaque fois qu'il faut écarter le trafic légitime.
DOMAINES_WHITELIST = {
    "windowsupdate.com",
    "ntp.org",
    "av-updates.example.net",
    "sync-backup.example.net",
    "metrics-updates.example.net",
}

### 1.5 Détection du beacon

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4</legend>

Trouvez le couple `(src_ip, domain)` de CV minimal dans `cv_par_couple`, en écartant les domaines de `DOMAINES_WHITELIST`. Rangez-le dans les variables `src_beacon` et `domaine_beacon`.

</fieldset>

In [ ]:
# BEGIN
## Rappel : `df.sort_values("col")` trie, `df.iloc[0]` donne la première ligne.
# src_beacon = ...
# domaine_beacon = ...
hors_whitelist = cv_par_couple[~cv_par_couple["domain"].isin(DOMAINES_WHITELIST)]
suspect = hors_whitelist.sort_values("cv").iloc[0]
# Certains domaines légitimes (mises à jour, NTP…) sont *aussi* périodiques :
# on les écarte avant de chercher le CV minimal (cf. `hors_whitelist`).
src_beacon, domaine_beacon = suspect["src_ip"], suspect["domain"]
# END

In [ ]:
assert empreinte((str(src_beacon), str(domaine_beacon))) == "21a1cf98d10e", f"trouvé : {src_beacon} -> {domaine_beacon}"
print("ex 4 OK :", src_beacon, "->", domaine_beacon)

On affiche les couples les plus réguliers (CV < 0,05), triés par domaine :

In [ ]:
cv_par_couple[cv_par_couple["cv"] < 0.05].sort_values(["domain", "src_ip"])

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5</legend>

On lit le tableau ci-dessus (couples de CV faible, triés par domaine).

1. Quelle autre caractéristique distingue le couple `(src_beacon, domaine_beacon)`
   des mises à jour de ce tableau ? Pourquoi est-ce suspect ?
2. Son CV (≈ 0,01) est plus élevé que celui des mises à jour (≈ 0,003), alors que
   le jitter est du même ordre (quelques secondes) des deux côtés : pourquoi ?
   (Indice : regardez `n`, sur une fenêtre de 12 h.)

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. `10.0.0.66` est le **seul hôte** à contacter `sync-updates.example.net`, alors
   que chaque mise à jour est contactée par 6 à 11 hôtes. Un service légitime est
   déployé sur tout le parc ; un implant ne tourne que sur la machine compromise.
2. Le CV divise l'écart-type par la moyenne. Le beacon a 102 écarts sur 12 h, soit
   un intervalle moyen d'environ 420 s ; les mises à jour en ont 23 à 35, soit
   1200 à 1800 s. Pour un même jitter (±5 s), un intervalle court donne un CV plus
   grand. Il reste très loin des CV de la navigation humaine (≥ 0,7).

</fieldset>

### 1.6 Extraction des IOC

Le couple (hôte, domaine) suspect étant identifié, on extrait ses **IOC**
(indicateurs de compromission) : les IP de destination distinctes
contactées par cet hôte pour ce domaine. La même extraction resservira pour
chaque hôte compromis du TD : on l'écrit donc une fois pour toutes, sous forme de fonction.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6</legend>

Complétez `destinations_distinctes` : à partir d'un dataframe de connexions (mêmes
colonnes que `df_logs_reseau`), elle renvoie les destinations distinctes : une ligne
par IP de destination, colonnes `dst_ip` et `domain`, triées par `dst_ip`, index
renuméroté `0, 1, 2…`.

</fieldset>

In [ ]:
def destinations_distinctes(connexions: pd.DataFrame) -> pd.DataFrame:
    """Destinations distinctes (`dst_ip`, `domain`) de `connexions`, triées par `dst_ip`."""
    # BEGIN
    ## Indice : `df.reset_index(drop=True)` renumérote l'index à partir de 0.
    # raise NotImplementedError
    return (
        connexions[["dst_ip", "domain"]]
        .drop_duplicates("dst_ip")
        .sort_values("dst_ip")
        .reset_index(drop=True)
    )
    # END

In [ ]:
def check_destinations_distinctes():
    # 3 connexions vers 2.2.2.2 (dont une dupliquée) et 1.1.1.1 : 2 destinations.
    petit = pd.DataFrame({
        "src_ip": ["a", "a", "a"],
        "dst_ip": ["2.2.2.2", "1.1.1.1", "2.2.2.2"],
        "domain": ["y.net", "x.net", "y.net"],
        "bytes_out": [10, 20, 30],
    })
    attendu = pd.DataFrame({"dst_ip": ["1.1.1.1", "2.2.2.2"], "domain": ["x.net", "y.net"]})
    res = destinations_distinctes(petit)
    assert list(res.columns) == ["dst_ip", "domain"], "colonnes attendues : dst_ip, domain"
    assert list(res.index) == [0, 1], "index à renuméroter (reset_index)"
    assert res.equals(attendu), f"résultat incorrect :\n{res}"
    print("ex 6 OK")

check_destinations_distinctes()

On extrait ici les IOC du beacon : les destinations du couple `(src_beacon, domaine_beacon)`.

In [ ]:
ioc_beacon = destinations_distinctes(
    df_logs_reseau[(df_logs_reseau["src_ip"] == src_beacon) & (df_logs_reseau["domain"] == domaine_beacon)]
)
assert empreinte(list(ioc_beacon["dst_ip"])) == "e4c7de31dd5c", "IP de destination incorrectes"
assert empreinte(list(ioc_beacon["domain"])) == "e55946171a33", "domaines incorrects"
print("IOC du beacon :", len(ioc_beacon))
ioc_beacon

### 1.7 Sans liste blanche : la stabilité du payload

La liste blanche a un défaut : il faut la tenir à jour, et un attaquant peut
choisir un nom de domaine qui ressemble à un service légitime. Un second
critère ne dépend d'aucune liste : la **taille des messages** (`bytes_out`).
Un beacon envoie à chaque fois le même petit message (« je suis là, des
ordres ? ») ; un téléchargement de mise à jour, lui, varie d'une fois sur l'autre.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 7</legend>

Créez la liste triée `payload_constant` des couples `(src_ip, domain)` ayant au moins `N_MIN` **connexions** et dont `bytes_out` ne prend qu'**une seule valeur**. N'utilisez pas `DOMAINES_WHITELIST`.

</fieldset>

In [ ]:
# BEGIN
## Indice : `agg` accepte plusieurs colonnes de sortie, par ex.
## `.agg(n=("bytes_out", "count"), nb_tailles=("bytes_out", "nunique"))`
# payload_constant = ...
stats_tailles = df_logs_reseau.groupby(["src_ip", "domain"]).agg(
    n=("bytes_out", "count"), nb_tailles=("bytes_out", "nunique")
)
suspects = stats_tailles[(stats_tailles["n"] >= N_MIN) & (stats_tailles["nb_tailles"] == 1)]
payload_constant = sorted(suspects.index)
# END

In [ ]:
assert empreinte([tuple(map(str, c)) for c in payload_constant]) == "693be4bafa36", f"trouvé : {payload_constant}"
print("ex 7 OK :", payload_constant)

Après la vérification, la cellule suivante affiche, pour les couples
réguliers (CV < 0,1), le nombre de tailles distinctes envoyées.

In [ ]:
tailles = (df_logs_reseau.groupby(["src_ip", "domain"])["bytes_out"]
           .nunique().rename("nb_tailles").reset_index())
(cv_par_couple[cv_par_couple["cv"] < 0.1]
    .merge(tailles, on=["src_ip", "domain"])
    .sort_values("nb_tailles")
    .head(10))

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 8</legend>

1. Pourquoi ce critère trouve-t-il le beacon sans liste blanche ?
2. Citez un service légitime qui pourrait, lui aussi, envoyer des messages de taille constante. Que faut-il en conclure ?
3. Comment l'attaquant peut-il casser ce signal ? Quel signal lui reste-t-il alors à masquer ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. Les mises à jour sont périodiques (CV faible) mais leur taille varie à
   chaque connexion (`nb_tailles` = nombre de connexions) ; le beacon envoie
   toujours 512 octets (`nb_tailles` = 1). Les deux critères réunis (CV faible
   **et** taille constante) isolent le beacon.
2. Un *heartbeat* de supervision, une requête NTP (48 octets), un keepalive…
   Ce critère seul produit aussi des faux positifs : on **croise** les signaux
   (régularité, taille, domaine connu ou non).
3. En ajoutant un remplissage aléatoire (*padding*) à chaque message : la
   taille varie, `nb_tailles` explose. Il lui reste à casser la régularité
   (jitter, timing aléatoire) : voir 1.8 et la partie 2.

</fieldset>

### 1.8 Jitter : jusqu'où le CV tient-il ?

Pour paraître moins mécanique, un implant ajoute un aléa (*jitter*) à son
intervalle de rappel.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 9</legend>

Un implant rappelle son C2 toutes les 60 s, avec un jitter de ±30 % : chaque intervalle est tiré au hasard, uniformément, entre 42 s et 78 s.

1. Les écarts à la moyenne vont de 0 à 18 s. Donnez un ordre de grandeur de l'écart-type, puis du CV.
2. Comparez aux CV du tableau de la partie 1.3 (mises à jour ≈ 0,003, beacon ≈ 0,01, navigation humaine ≥ 0,7). L'implant reste-t-il détectable ? Proposez un seuil, et dites ce qu'il coûte.
3. L'implant passe à un timing **totalement aléatoire** (intervalle de loi exponentielle, CV ≈ 1). Que peut encore observer le défenseur ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. Écart-type de l'ordre de 10 s (entre 0 et 18, un peu au-dessus du milieu),
   donc CV ≈ 10 / 60 ≈ **0,15–0,2** (valeur exacte : 0,3 / √3 ≈ 0,17).
2. Oui : 0,17 reste très en dessous des CV humains (≥ 0,7). Un seuil vers
   CV < 0,3 le détecte encore, mais laisse passer davantage de trafic
   légitime un peu régulier (plus de faux positifs à trier). Plus le jitter
   augmente, plus le CV se rapproche de celui d'un humain.
3. Le rythme ne trahit plus rien. Il reste **où** l'implant se connecte :
   une plage d'IP que personne d'autre ne contacte (partie 2), des
   destinations rares (partie 4), et le volume de connexions.

</fieldset>

On vérifie ici par simulation (200 intervalles tirés au hasard) :

In [ ]:
rng = random.Random(0)
avec_jitter = [rng.uniform(42, 78) for _ in range(200)]
aleatoire = [rng.expovariate(1 / 60) for _ in range(200)]
for nom, intervalles in [("jitter ±30 %", avec_jitter), ("timing aléatoire", aleatoire)]:
    print(f"{nom:17} : CV = {statistics.stdev(intervalles) / statistics.mean(intervalles):.2f}")

## 2. Timing aléatoire : il ne reste que la plage d'IP

L'implant le plus sophistiqué **randomise totalement** son timing : plus aucun
rythme à détecter. Mais il doit toujours joindre son serveur C2, hébergé dans un
`/24` que l'entreprise ne contacte jamais.

**Méthode :**

1. Écarter le domaine du beacon trouvé en partie 1 (`domaine_beacon`).
2. Réduire chaque IP restante à son préfixe `/24`.
3. Pour chaque préfixe, calculer **deux chiffres séparés** :
   - `nb_sources` : combien d'hôtes distincts le touchent ;
   - `nb_connexions` : combien de connexions il reçoit.
4. Un préfixe **rare** (1 seule source) **et massivement contacté** = suspect.

**Note :** cet implant ne laisse aucun rythme exploitable et ajoute un
*padding* aléatoire à ses messages : ni la régularité, ni la stabilité
du payload (partie 1), ni le comptage de destinations (partie 3, plus loin) ne le détectent.
Seule la plage d'IP le trahit.

### 2.1 Préfixe /24 d'une IP

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 10</legend>

Complétez `prefixe` : elle renvoie le préfixe `/24` d'une IPv4, c'est-à-dire ses trois premiers octets (par exemple `"185.165.29"` pour `"185.165.29.10"`).

</fieldset>

In [ ]:
def prefixe(ip: str) -> str:
    """Préfixe /24 d'une IPv4."""
    # BEGIN
    # raise NotImplementedError
    return ".".join(ip.split(".")[:3])
    # END

In [ ]:
assert prefixe("185.165.29.10") == "185.165.29"
assert prefixe("10.0.0.1") == "10.0.0"
print("ex 10 OK")

In [ ]:
# DELETE
df_logs_reseau.assign(prefix=df_logs_reseau["dst_ip"].map(prefixe))

### 2.2 Hôtes sur une plage rare et massivement contactée

Pour chaque préfixe `/24` des destinations, on compte le nombre de sources
distinctes qui le contactent, ainsi que le nombre total de requêtes qu'il reçoit.
On ne garde ensuite que les préfixes à une seule source et massivement contactés,
puis on remonte aux hôtes sources correspondants.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 11a</legend>

1. Créez `df_sans_beacon` : les connexions de `df_logs_reseau` **hors** domaine
   du beacon `domaine_beacon` (trouvé en partie 1, lui aussi sur un /24 rare), avec
   une colonne `prefix` en plus : le préfixe /24 de `dst_ip`.
2. Créez `stats_prefixes` : un dataframe indexé par `prefix`, avec deux colonnes
   `nb_sources` (nombre d'hôtes **distincts**) et `nb_connexions` (nombre de connexions).

</fieldset>

In [ ]:
# BEGIN
## Indice : `serie.map(prefixe)` ; `agg` à deux sorties nommées (voir le préambule).
# df_sans_beacon = ...
# stats_prefixes = ...
df_sans_beacon = df_logs_reseau[df_logs_reseau["domain"] != domaine_beacon].copy()
df_sans_beacon["prefix"] = df_sans_beacon["dst_ip"].map(prefixe)
stats_prefixes = df_sans_beacon.groupby("prefix").agg(
    nb_sources=("src_ip", "nunique"), nb_connexions=("src_ip", "count")
)
# END

In [ ]:
assert "prefix" in df_sans_beacon.columns, "colonne prefix attendue"
assert not (df_sans_beacon["domain"] == domaine_beacon).any(), "le domaine du beacon doit être écarté"
assert stats_prefixes.index.name == "prefix", "index attendu : prefix"
assert {"nb_sources", "nb_connexions"} <= set(stats_prefixes.columns), f"colonnes attendues : nb_sources, nb_connexions ; obtenues : {list(stats_prefixes.columns)}"
assert empreinte((int(stats_prefixes["nb_sources"].sum()), int(stats_prefixes["nb_connexions"].sum()))) == "d8d06e7660f5", "statistiques incorrectes"
print("ex 11a OK :", len(stats_prefixes), "préfixes")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 11b</legend>

Créez la liste triée `hotes_plage_rare` des hôtes source (de `df_sans_beacon`) qui
touchent un préfixe de `stats_prefixes` ayant **une seule** source et au moins
`VOL_MIN` connexions.

</fieldset>

In [ ]:
VOL_MIN = 50  # nombre minimal de connexions vers le /24

# BEGIN
# hotes_plage_rare = ...
rares = stats_prefixes[(stats_prefixes["nb_sources"] == 1) & (stats_prefixes["nb_connexions"] >= VOL_MIN)].index
hotes_plage_rare = sorted(df_sans_beacon[df_sans_beacon["prefix"].isin(rares)]["src_ip"].unique())
# END

In [ ]:
assert empreinte([str(h) for h in hotes_plage_rare]) == "4c719326ff97", f"trouvé : {hotes_plage_rare}"
print("ex 11b OK :", hotes_plage_rare)

### 2.3 Extraction des IOC

On réutilise `destinations_distinctes` sur toutes les connexions de l'hôte suspect `hotes_plage_rare[0]` (tous domaines confondus).

In [ ]:
ioc_plage = destinations_distinctes(df_logs_reseau[df_logs_reseau["src_ip"] == hotes_plage_rare[0]])
assert ioc_plage["dst_ip"].is_unique
assert empreinte(list(ioc_plage["dst_ip"])) == "aff3643cf181", "IP de destination incorrectes"
print("IOC :", len(ioc_plage))
ioc_plage

## 3. Fast-flux : la destination qui tourne

Certains C2 changent d'IP **et** de domaine à chaque connexion pour échapper aux
filtres : c'est le **fast-flux** (ou DGA). Chaque domaine n'apparaissant qu'une
fois, grouper par domaine est inutile.

**Méthode :** regarder l'hôte *source*. Un hôte légitime contacte un petit nombre
de destinations. Un hôte en fast-flux contacte un grand nombre d'IP **distinctes**.

### 3.1 Hôte aux destinations multiples

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 12</legend>

Rangez dans la variable `hote_fastflux` l'hôte source qui contacte le plus d'IP de destination distinctes.

</fieldset>

In [ ]:
# BEGIN
# hote_fastflux = ...
hote_fastflux = df_logs_reseau.groupby("src_ip")["dst_ip"].nunique().sort_values(ascending=False).index[0]
# END

In [ ]:
assert empreinte(str(hote_fastflux)) == "c2af775133af", f"trouvé : {hote_fastflux}"
print("ex 12 OK :", hote_fastflux)

### 3.2 Extraction des IOC

On réutilise `destinations_distinctes` sur les connexions de `hote_fastflux`.

In [ ]:
ioc_fastflux = destinations_distinctes(df_logs_reseau[df_logs_reseau["src_ip"] == hote_fastflux])
assert len(ioc_fastflux) == 40
assert ioc_fastflux["dst_ip"].is_unique
print("IOC :", len(ioc_fastflux))
ioc_fastflux

## 4. Quand le comptage ne suffit plus

Le réseau grossit : on observe désormais une centaine d'hôtes, qui contactent
chacun des centaines d'IP de destination distinctes (CDN, edge, caches…).

Un hôte compromis renouvelle ses destinations à chaque connexion : compter ses
IP distinctes ne le distingue plus des autres. Mais ses destinations sont
**rares** : aucun autre hôte du réseau ne les contacte. On compte donc, pour
chaque hôte, le nombre d'IP qu'il est **seul** à contacter.

### 4.1 Chargement du grand réseau

In [ ]:
df_logs_reseau_grand = pd.read_csv("logs_reseau_grand.csv", parse_dates=["timestamp"])
print(df_logs_reseau_grand.shape)
df_logs_reseau_grand.head()

### 4.2 Lire la chaîne des destinations rares

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 13</legend>

On considère la chaîne suivante, appliquée au petit dataframe `petit` ci-dessous :

```python
nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
ip_rares = nb_sources[nb_sources == 1].index
rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
resultat = rare_par_hote.sort_values(ascending=False).index[0]
```

| | src_ip | dst_ip |
| --- | --- | --- |
| 0 | 10.0.0.1 | 198.18.0.1 |
| 1 | 10.0.0.2 | 198.18.0.1 |
| 2 | 10.0.0.3 | 198.18.0.1 |
| 3 | 10.0.0.1 | 198.18.0.2 |
| 4 | 10.0.0.2 | 198.18.0.2 |
| 5 | 10.0.0.3 | 193.29.0.7 |
| 6 | 10.0.0.3 | 193.29.0.8 |
| 7 | 10.0.0.2 | 185.220.3.4 |
| 8 | 10.0.0.3 | 193.29.0.7 |

1. Décrivez en français ce que contiennent `nb_sources`, `ip_rares` et `rare_par_hote` (que représentent l'index et les valeurs ?).
2. Calculez à la main leur valeur sur `petit`, ainsi que `resultat`.
3. À la dernière ligne, que changerait `count()` à la place de `nunique()` ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

1. - `nb_sources` : pour chaque IP de destination (index), le nombre d'hôtes
     sources **distincts** qui la contactent.
   - `ip_rares` : la liste des IP de destination contactées par **un seul** hôte.
   - `rare_par_hote` : pour chaque hôte source (index), le nombre d'IP rares
     distinctes qu'il contacte. `resultat` est l'hôte qui en contacte le plus.
2. - `nb_sources` : `198.18.0.1` → 3, `198.18.0.2` → 2, `193.29.0.7` → 1,
     `193.29.0.8` → 1, `185.220.3.4` → 1.
   - `ip_rares` : `193.29.0.7`, `193.29.0.8`, `185.220.3.4`.
   - `rare_par_hote` : `10.0.0.2` → 1, `10.0.0.3` → 2 (`10.0.0.1` n'apparaît
     pas : il ne touche aucune IP rare).
   - `resultat` : `"10.0.0.3"`.
3. `count()` compte les **connexions** : `10.0.0.3` contacte deux fois
   `193.29.0.7`, il aurait 3 au lieu de 2. Avec `nunique()`, on compte des
   **destinations différentes** : un hôte qui contacte souvent la même IP rare
   ne pèse pas plus qu'un hôte qui la contacte une fois.

</fieldset>

On vérifie ici en exécutant la chaîne sur `petit` :

In [ ]:
petit = pd.DataFrame({
    "src_ip": ["10.0.0.1", "10.0.0.2", "10.0.0.3", "10.0.0.1", "10.0.0.2",
               "10.0.0.3", "10.0.0.3", "10.0.0.2", "10.0.0.3"],
    "dst_ip": ["198.18.0.1", "198.18.0.1", "198.18.0.1", "198.18.0.2", "198.18.0.2",
               "193.29.0.7", "193.29.0.8", "185.220.3.4", "193.29.0.7"],
})
df = petit
nb_sources = df.groupby("dst_ip")["src_ip"].nunique()
ip_rares = nb_sources[nb_sources == 1].index
rare_par_hote = df[df["dst_ip"].isin(ip_rares)].groupby("src_ip")["dst_ip"].nunique()
print(nb_sources, end="\n\n")
print(list(ip_rares), end="\n\n")
print(rare_par_hote, end="\n\n")
print("resultat :", rare_par_hote.sort_values(ascending=False).index[0])

### 4.3 Détection de l'hôte compromis

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 14</legend>

En appliquant la chaîne de l'exercice 13 au grand réseau (`df_logs_reseau_grand`), rangez dans la variable `hote_compromis` l'hôte qui contacte le plus d'IP rares.

</fieldset>

In [ ]:
# BEGIN
# hote_compromis = ...
nb_sources = df_logs_reseau_grand.groupby("dst_ip")["src_ip"].nunique()
ip_rares = nb_sources[nb_sources == 1].index
rare_par_hote = (df_logs_reseau_grand[df_logs_reseau_grand["dst_ip"].isin(ip_rares)]
                 .groupby("src_ip")["dst_ip"].nunique())
hote_compromis = rare_par_hote.sort_values(ascending=False).index[0]
# END

In [ ]:
assert empreinte(str(hote_compromis)) == "6fd133d0c5c9", f"trouvé : {hote_compromis}"
print("ex 14 OK :", hote_compromis)

### 4.4 Extraction des IOC

On réutilise `destinations_distinctes` sur les connexions de `hote_compromis`.

In [ ]:
ioc_grand = destinations_distinctes(df_logs_reseau_grand[df_logs_reseau_grand["src_ip"] == hote_compromis])
assert len(ioc_grand) == 400
assert ioc_grand["dst_ip"].is_unique
print("IOC :", len(ioc_grand))
ioc_grand

## Bilan

| cas d'étude | signal | notion pandas | hôte trouvé |
| --- | --- | --- | --- |
| 1 | régularité temporelle | `groupby` + `agg(list)` + CV | `10.0.0.66` |
| 1 | stabilité du payload | `groupby().agg(nunique)` | `10.0.0.66` |
| 2 | plage d'IP rare et concentrée | `map` + deux `groupby` | `10.0.0.68` |
| 3 | multiplicité des destinations | `groupby().nunique()` | `10.0.0.67` |
| 4 | destinations rares (une seule source) | `nunique` + `isin` | `10.0.0.167` |

**À retenir :** la détection de C2 est **heuristique** — on croise plusieurs
signaux (rythme, destinations, plage d'IP, rareté des destinations). Aucun
signal seul ne suffit, et chaque contre-mesure de l'attaquant (jitter,
padding aléatoire, fast-flux, timing aléatoire, destinations partagées) casse
un signal… mais en laisse un autre.